# Cross-Participant EEGBCI Decoding

This is a companion tutorial to the paper *"A Primer on Low-Dimensional Neural Dynamics: PCA-Based Trajectory Analysis for EEG and MEG."*

The trajectory tutorials showed how left- and right-hand movements evolve as paths through neural state space. This notebook asks a stricter, predictive question:

> **Can left- versus right-hand execution be decoded in a participant who contributed no labeled trials to classifier training?**

That question is the honest test of a low-dimensional representation. Group-mean trajectory separation tells you the conditions *differ on average*; single-trial cross-participant decoding tells you whether that difference is usable in a brain that the model has never seen.

<div class="alert alert-secondary">
<b>🗺️ Analysis roadmap:</b><br>
<ol style="margin-bottom: 0; margin-top: 5px;">
  <li>Define the predictive question and the inferential unit.</li>
  <li>Load the native sensor-time representation.</li>
  <li>Verify target encoding and participant-level class balance.</li>
  <li>Declare leave-one-participant-out cross-validation.</li>
  <li>Configure fold-local sliding temporal decoding.</li>
  <li>Build five representations of the same data.</li>
  <li>Run and audit every outer fold.</li>
  <li>Inspect time-resolved generalization and gain through time.</li>
  <li>Examine held-out-participant variability.</li>
  <li>Test the differences with a paired permutation test.</li>
  <li>Bound the alignment result with three validations.</li>
  <li>Export the full analysis and structured HTML report.</li>
</ol>
</div>

### The five representations

Every representation below feeds the *identical* classifier, folds, scaling, metric, and time axis. Only the feature space changes, which is what makes the comparison interpretable.

| Representation | Basis | Per-participant? | Rotated onto a shared template? |
|---|---|---|---|
| **Sensors** | 64 EEG channels | — | — |
| **Shared PCA** | one PCA on pooled training participants | no | — |
| **Unaligned PCA** | one PCA per participant | yes | **no** |
| **Aligned PCA (3C / 30C)** | one PCA per participant | yes | **yes** |

The pairing of the last two is the point. *Unaligned PCA* and *Aligned PCA* differ by exactly one operation — the Procrustes rotation — so any difference between them isolates what the rotation contributes, rather than what merely having a participant-specific basis contributes.

## 0. Setup & Configuration

Every scientific choice is declared before a single trial is loaded. The notebook performs the complete analysis in its own cells; only the report renderer is shared with the companion script, so both entry points produce exactly the same structured HTML output.

### 0.1. Environment & imports

`coco-pipe` owns the cross-validation engine, the fold-local preprocessing, the temporal estimator, the alignment transformer, the result containers, the statistics, and the plotting. The repository helpers only load the prepared EEGBCI epochs and record provenance.

In [25]:
# --- Standard library -------------------------------------------------------
import os
import warnings
from pathlib import Path

# --- Numerical and plotting libraries ---------------------------------------
import numpy as np
import pandas as pd
from IPython.display import display

# --- coco-pipe decoding ------------------------------------------------------
from coco_pipe.decoding import (
    ChanceAssessmentConfig,
    CVConfig,
    Experiment,
    ExperimentConfig,
    ReducerConfig,
    StatisticalAssessmentConfig,
    TemporalAlignmentConfig,
    TemporalDecoderConfig,
)
from coco_pipe.decoding.configs import ClassicalModelConfig
from coco_pipe.decoding.stats import run_paired_permutation_assessment
from coco_pipe.transforms import TemporalProcrustesAlignment
from coco_pipe.viz.interactive import (
    plot_distribution_groups,
    plot_group_scatter_with_mean,
    plot_heatmap,
)
from coco_pipe.viz.interactive.decoding import (
    plot_temporal_score_curve,
    plot_temporal_statistical_assessment,
)
from coco_pipe.viz.theme import set_coco_theme

# --- Repository helpers ------------------------------------------------------
from pca_neural_trajectories import (
    LABEL_NAMES,
    contrast_label,
    facet_figures,
    load_eegbci_container,
    setup_data_bids,
    write_manifest,
)


### 0.2. Analysis parameters & output contract

The tutorial defaults to the first ten sampling-rate-compatible participants, which runs in a few minutes on a laptop. Participants S088, S092 and S100 are excluded throughout: they were recorded at 128 Hz instead of 160 Hz (a documented EEGBCI anomaly), so their epochs do not share a time axis with the rest of the cohort.

<div class="alert alert-info">
<b>⚙️ Environment overrides:</b><br>
Use <code>EEG_N_SUBJECTS</code>, <code>EEG_DECODING_N_JOBS</code>, <code>EEG_DECODING_PERMUTATIONS</code>, <code>EEG_BIDS_ROOT</code> and <code>EEG_DECODING_OUTPUT</code> to change runtime settings without editing analysis cells. Set <code>EEG_PREPARE_DATA=1</code> only when preprocessing should be requested explicitly.
</div>

In [26]:
set_coco_theme(mode="paper", colorblind=True)

SEED = 42
CONDITIONS = (3, 4)                    # 3 = left hand, 4 = right hand (execution)
ANALYSIS_WINDOW = (-0.2, 1.0)
CHANCE_LEVEL = 1.0 / len(CONDITIONS)
COMPARISON_CONTRASTS = ((3, 4), (5, 6), (7, 9), (3, 4, 5, 6))
# Execution, imagery, imagery-vs-execution, and 4-class
RUN_CONTRAST_COMPARISON = os.getenv("EEG_CONTRAST_COMPARISON", "1") == "1"
N_COMPONENTS = int(os.getenv("EEG_DECODING_COMPONENTS", "30"))
SMALL_N_COMPONENTS = int(os.getenv("EEG_DECODING_SMALL_COMPONENTS", "3"))
N_PERMUTATIONS = int(os.getenv("EEG_DECODING_PERMUTATIONS", "200"))
WITHIN_SUBJECT_SPLITS = 5
GEOMETRY_MAX_TRIALS = 200
N_JOBS = int(os.getenv("EEG_DECODING_N_JOBS", "-1"))
N_SUBJECTS = int(os.getenv("EEG_N_SUBJECTS", "10"))
BIDS_ROOT = Path(os.getenv("EEG_BIDS_ROOT", "PhysioNet_EEGBCI/BIDS"))
OUTPUT = Path(os.getenv("EEG_DECODING_OUTPUT", "outputs/tutorial_eegbci_decoding"))
FIGURES_DIR = OUTPUT / "figures"
RESULTS_DIR = OUTPUT / "experiment_results"
PREPARE_DATA = os.getenv("EEG_PREPARE_DATA", "0") == "1"

# Participants recorded at 128 Hz rather than 160 Hz.
EXCLUDED_SUBJECTS = {88, 92, 100}
available = [subject for subject in range(1, 110) if subject not in EXCLUDED_SUBJECTS]
SUBJECTS_REQUESTED = available[:N_SUBJECTS]
SUBJECTS = tuple(f"{subject:03d}" for subject in SUBJECTS_REQUESTED)

SHARED_PCA = f"Shared PCA ({N_COMPONENTS})"
UNALIGNED_PCA = f"Unaligned PCA ({N_COMPONENTS}C)"
ALIGNED_SMALL = f"Aligned PCA ({SMALL_N_COMPONENTS}C)"
ALIGNED_LARGE = f"Aligned PCA ({N_COMPONENTS}C)"
CALIBRATION = f"Aligned PCA ({N_COMPONENTS}C, calibration)"
REPRESENTATION_COLORS = {
    "Sensors": "#1b9e77",
    SHARED_PCA: "#e6ab02",
    UNALIGNED_PCA: "#d95f02",
    ALIGNED_SMALL: "#7570b3",
    ALIGNED_LARGE: "#2a78d6",
    CALIBRATION: "#c51b7d",
    "Within participant (sensors)": "#444444",
}

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Participants requested: {SUBJECTS}")
print(f"BIDS input:    {BIDS_ROOT}")
print(f"Output bundle: {OUTPUT}")

Participants requested: ('001', '002', '003', '004', '005', '006', '007', '008', '009', '010')
BIDS input:    PhysioNet_EEGBCI/BIDS
Output bundle: outputs/tutorial_eegbci_decoding


## Step 1. Define the Predictive Question

The question is **cross-participant generalization**, not whether a model can classify held-out trials from a familiar participant. That distinction determines the entire validation design, so it is worth stating in full before any code runs.

- **Observation:** one epoched movement trial.
- **Target:** left hand (`0`) versus right hand (`1`), executed movement.
- **Inferential unit:** the participant.
- **Generalization target:** a participant whose labeled trials were entirely absent during training.

<div class="alert alert-danger">
<b>🚫 Why random trial splitting is invalid here:</b><br>
Trials from one participant share anatomy, electrode placement, impedance, preprocessing history and stable idiosyncratic signal structure. If some of a participant's trials are in training and others in test, the classifier can score well by recognizing <i>the participant</i> rather than by learning a pattern that transfers. The reported number would then answer a question nobody asked.
</div>

## Step 2. Load the Native Sensor-Time Representation

`coco-pipe` accepts the original `(trial, channel, time)` array directly, so no custom time-binning helper is needed. We load the two execution conditions over −0.2–1.0 s and apply the pre-cue −0.2–0.0 s sensor baseline.

Note what is *not* done here: no global standardization, no global PCA, no channel selection. Anything fitted on data is fitted later, inside a fold, after the participants have been separated. A transformation applied now would see every participant at once, including the one we are about to pretend we have never met.

<div class="alert alert-warning">
<b>🧠 Preprocessing boundary:</b><br>
The introductory tutorial prepares the BIDS derivatives (bad-channel interpolation, common-average reference, ICA removal of blink and cardiac components, 0.5–40 Hz bandpass). This notebook does not silently download or preprocess data; set <code>PREPARE_DATA=True</code> only when that external work is explicitly intended.
</div>

In [27]:
if PREPARE_DATA:
    setup_data_bids(
        subjects=SUBJECTS_REQUESTED,
        runs=list(range(3, 15)),
        root=BIDS_ROOT,
    )
if not BIDS_ROOT.exists():
    raise FileNotFoundError(
        f"EEGBCI BIDS data were not found at {BIDS_ROOT}. "
        "Run the introductory tutorial first."
    )

container = load_eegbci_container(
    BIDS_ROOT,
    subjects=SUBJECTS,
    runs=tuple(range(3, 15)),
    conditions=CONDITIONS,
    tmin=ANALYSIS_WINDOW[0],
    tmax=ANALYSIS_WINDOW[1],
    baseline=(-0.2, 0.0),
)
X = np.asarray(container.X, dtype=np.float32)
times = np.asarray(container.coords["time"], dtype=float)
condition = np.asarray(container.y, dtype=int)
subject_ids = np.asarray(container.coords["subject"]).astype(str)
trial_ids = np.asarray(container.ids).astype(str)

condition_to_class = {value: index for index, value in enumerate(CONDITIONS)}
y = np.array([condition_to_class[value] for value in condition])
TARGET_NAMES = [LABEL_NAMES[value] for value in CONDITIONS]

ANALYZED_SUBJECTS = sorted(np.unique(subject_ids).tolist())
if len(ANALYZED_SUBJECTS) < 3:
    raise RuntimeError("Cross-participant decoding needs at least three participants.")
if N_COMPONENTS > X.shape[1]:
    raise ValueError(f"N_COMPONENTS={N_COMPONENTS} exceeds {X.shape[1]} channels.")

Reading 0 ... 19999  =      0.000 ...   124.994 secs...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
NOTE: pick_ty

In [28]:
data_summary = pd.Series({
    "trials": X.shape[0],
    "channels": X.shape[1],
    "time samples": X.shape[2],
    "participants": len(ANALYZED_SUBJECTS),
    "first time (s)": float(times[0]),
    "last time (s)": float(times[-1]),
    "target": " vs ".join(TARGET_NAMES),
})
data_summary.to_frame("value")

,value
trials,450
channels,64
time samples,193
participants,10
first time (s),-0.2
last time (s),1.0
target,Left Hand (Exec) vs Right Hand (Exec)


## Step 3. Verify the Target and Participant-Level Class Balance

The target encoding is fixed before model fitting: condition 3 becomes left hand (`0`) and condition 4 becomes right hand (`1`). We then document the number of trials in every participant × class cell.

Balanced accuracy is the mean recall across classes, so each class contributes equally to the score even when a participant has modestly unequal trial counts. The classifier additionally receives `class_weight='balanced'`, and those weights are estimated from the **training fold only**.

<div class="alert alert-success">
<b>✅ No test-set resampling:</b><br>
Class balance is handled by the metric and by training-fold weights. Test trials are never duplicated, removed, or used to choose a training weight — a trial's own existence must not influence how it is scored.
</div>

In [29]:
trial_counts = (
    pd.DataFrame({
        "subject": subject_ids,
        "condition": condition,
        "condition_name": [LABEL_NAMES[value] for value in condition],
    })
    .groupby(["subject", "condition", "condition_name"], as_index=False)
    .size()
    .rename(columns={"size": "n_trials"})
)
display(trial_counts.head(10))

trial_counts.pivot(index="subject", columns="condition_name", values="n_trials")

,subject,condition,condition_name,n_trials
0,001,3,Left Hand (Exec),23
1,001,4,Right Hand (Exec),22
2,002,3,Left Hand (Exec),23
3,002,4,Right Hand (Exec),22
4,003,3,Left Hand (Exec),22
5,003,4,Right Hand (Exec),23
6,004,3,Left Hand (Exec),22
7,004,4,Right Hand (Exec),23
8,005,3,Left Hand (Exec),23
9,005,4,Right Hand (Exec),22


condition_name,Left Hand (Exec),Right Hand (Exec)
subject,,
001,23,22
002,23,22
003,22,23
004,22,23
005,23,22
006,22,23
007,23,22
008,22,23
009,23,22


## Step 4. Declare Leave-One-Participant-Out Cross-Validation

`CVConfig(strategy='leave_one_group_out')` creates one outer fold per participant. Every trial from the held-out participant becomes test data; every trial from all other participants becomes training data.

The same participant groups are passed to every representation, so the resulting scores are naturally **paired by held-out participant** — the property that later lets us test differences rather than merely eyeball two curves. We verify the generated splits after fitting rather than trusting that the configuration was applied as intended.

In [30]:
cross_val = CVConfig(strategy="leave_one_group_out", shuffle=False)

## Step 5. Configure Fold-Local Sliding Temporal Decoding

At each latency, `wrapper='sliding'` fits a separate logistic-regression classifier on the feature values at that one time point. The result is a time-resolved generalization curve rather than a single number, which lets us ask *when* the discriminative information appears.

`use_scaler=True` instructs `coco-pipe` to fit standardization inside each training fold; the held-out participant is transformed with those training-fold parameters. Fitting a single `StandardScaler` on all participants before splitting would quietly leak the test distribution into the training pipeline.

<div class="alert alert-warning">
<b>📈 Multiple-latency caution:</b><br>
Every plotted time point is a separately fitted model. The highest point is a useful descriptive summary, but it is not automatically a family-wise-error-corrected result. Step 10 supplies the corrected test.
</div>

In [31]:
decoder = TemporalDecoderConfig(
    wrapper="sliding",
    base=ClassicalModelConfig(
        estimator="LogisticRegression",
        params={"class_weight": "balanced", "max_iter": 2000},
    ),
    n_jobs=1,
    verbose=False,
)
sensor_config = ExperimentConfig(
    task="classification",
    models={"Logistic regression": decoder},
    metrics=["balanced_accuracy"],
    cv=cross_val,
    use_scaler=True,
    random_state=SEED,
    n_jobs=N_JOBS,
    verbose=False,
)
sensor_config

ExperimentConfig(task='classification', output_dir=None, tag='experiment', random_state=42, models={'Logistic regression': TemporalDecoderConfig(kind='temporal', wrapper='sliding', base=ClassicalModelConfig(kind='classical', method='ClassicalModel', estimator='LogisticRegression', params={'class_weight': 'balanced', 'max_iter': 2000}, input_kind='tabular'), scoring=None, n_jobs=1, position=0, allow_2d=False, verbose=False)}, grids=None, cv=CVConfig(strategy='leave_one_group_out', n_splits=5, shuffle=False, random_state=42, test_size=0.2, stratify=False, group_key=None, auto_reduce_n_splits=True), tuning=TuningConfig(enabled=False, search_type='grid', n_iter=10, scoring=None, n_jobs=-1, random_state=42, cv=None, allow_nongroup_inner_cv=False), feature_selection=FeatureSelectionConfig(enabled=False, method='sfs', n_features=None, direction='forward', tol=None, cv=None, scoring=None, allow_nongroup_inner_cv=False), erasure=ErasureConfig(enabled=False, method='leace', params={}), temporal_

## Step 6. Build Five Representations of the Same Data

Each representation copies the sensor configuration and changes exactly one field, so the feature space is the only experimental variable.

<span style="color: #0072B2;"><b>Shared PCA</b></span> — a single fold-local PCA fitted on the pooled training participants (`ReducerConfig`) and applied unchanged to everyone, including the held-out participant. There is no per-participant basis and no rotation. This is the classic "reduce dimensionality, then decode" baseline.

<span style="color: #0072B2;"><b>Unaligned PCA</b></span> — a separate PCA per participant, and nothing else. Each participant is described in their *own* principal axes. This is the control that decides the whole story, because it differs from the aligned variants by the rotation alone.

<span style="color: #0072B2;"><b>Aligned PCA</b></span> — a separate PCA per participant, followed by an orthogonal Procrustes rotation of that participant's grand-mean trajectory onto a **training-only** temporal template. Concretely, inside each fold:

1. fit a shared PCA on the training participants and average their projected trials over **all** trials, ignoring condition — producing a `(time × K)` template path;
2. fit a separate PCA on each participant's own data, held-out participant included;
3. rotate each participant's own grand-mean trajectory onto the template, and apply that same rotation to all of their trials.

After step 3, participant 3's "PC1" and participant 5's "PC1" mean the same thing *functionally*, rather than "whatever explained the most variance in that head". Note that the template is built without ever looking at a condition label, which is why the alignment can legitimately be estimated for a participant whose labels we are withholding.

We run alignment at two component counts. Three components is the number the trajectory figures live in; thirty is closer to the effective dimensionality of the data, and the pair reveals whether the low-dimensional picture is costing information.

<div class="alert alert-danger">
<b>🔒 Transductive scope:</b><br>
Aligned performance assumes an <i>unlabeled calibration batch</i> from the new participant — no labels are used, but some of their data is. This is standard domain adaptation and realistic in a BCI setting, but it is not "train on nine participants and apply blind to a single trial from the tenth". Step 11 quantifies exactly how much of the result depends on this.
</div>

In [32]:
shared_pca_config = sensor_config.model_copy(deep=True)
shared_pca_config.reducer = ReducerConfig(enabled=True, n_components=N_COMPONENTS)

# The control: identical per-participant PCA, with the rotation switched off.
unaligned_config = sensor_config.model_copy(deep=True)
unaligned_config.temporal_alignment = TemporalAlignmentConfig(
    enabled=True, n_components=N_COMPONENTS, adaptation="transductive", rotate=False
)

aligned_small_config = sensor_config.model_copy(deep=True)
aligned_small_config.temporal_alignment = TemporalAlignmentConfig(
    enabled=True, n_components=SMALL_N_COMPONENTS, adaptation="transductive"
)

aligned_large_config = sensor_config.model_copy(deep=True)
aligned_large_config.temporal_alignment = TemporalAlignmentConfig(
    enabled=True, n_components=N_COMPONENTS, adaptation="transductive"
)

# Validation variant (Step 11): each held-out participant's rotation is estimated
# from one half of their trials and applied to the other half.
calibration_config = sensor_config.model_copy(deep=True)
calibration_config.temporal_alignment = TemporalAlignmentConfig(
    enabled=True, n_components=N_COMPONENTS, adaptation="calibration"
)

experiments = {
    "Sensors": sensor_config,
    SHARED_PCA: shared_pca_config,
    UNALIGNED_PCA: unaligned_config,
    ALIGNED_SMALL: aligned_small_config,
    ALIGNED_LARGE: aligned_large_config,
}
REPRESENTATION_NAMES = list(experiments)
OTHER_REPRESENTATIONS = [name for name in REPRESENTATION_NAMES if name != "Sensors"]

pd.DataFrame([
    {
        "Representation": name,
        "shared_pca_components": (
            config.reducer.n_components if config.reducer.enabled else None
        ),
        "subject_pca_components": (
            config.temporal_alignment.n_components
            if config.temporal_alignment.enabled else None
        ),
        "rotated": (
            config.temporal_alignment.rotate
            if config.temporal_alignment.enabled else None
        ),
        "scaler_inside_fold": config.use_scaler,
        "cv": config.cv.strategy,
    }
    for name, config in {**experiments, CALIBRATION: calibration_config}.items()
])

,Representation,shared_pca_components,subject_pca_components,rotated,scaler_inside_fold,cv
0,Sensors,NaN,NaN,None,True,leave_one_group_out
1,Shared PCA (30),30.0,NaN,None,True,leave_one_group_out
2,Unaligned PCA (30C),NaN,30.0,False,True,leave_one_group_out
3,Aligned PCA (3C),NaN,3.0,True,True,leave_one_group_out
4,Aligned PCA (30C),NaN,30.0,True,True,leave_one_group_out
5,"Aligned PCA (30C, calibration)",NaN,30.0,True,True,leave_one_group_out


## Step 7. Run and Preserve Every Outer Fold

All six experiments receive the same epochs, labels, participant groups, trial identifiers and scientific time axis. `ExperimentResult` retains the fold scores, per-trial predictions, splits, fit diagnostics and configuration needed for a later audit.

We export the complete result objects rather than only the final mean curve. That is what makes Step 10 cheap: the permutation test re-scores stored predictions instead of refitting hundreds of temporal classifiers.

In [33]:
results = {}
temporal_frames = []
fold_frames = []
diagnostic_frames = []

for representation, config in {**experiments, CALIBRATION: calibration_config}.items():
    print(f"Running {representation} ...")
    result = Experiment(config).run(
        X,
        y,
        groups=subject_ids,
        sample_ids=trial_ids,
        observation_level="epoch",
        inferential_unit="subject",
        time_axis=times,
    )
    results[representation] = result

    temporal = result.get_temporal_score_summary()
    temporal["Estimator"] = temporal["Model"]
    temporal["Representation"] = representation
    temporal["Model"] = representation
    temporal_frames.append(temporal)

    folds = result.get_detailed_scores()
    folds["Estimator"] = folds["Model"]
    folds["Representation"] = representation
    fold_frames.append(folds)

    diagnostics = result.get_fit_diagnostics()
    diagnostics["Estimator"] = diagnostics["Model"]
    diagnostics["Representation"] = representation
    diagnostic_frames.append(diagnostics)

    slug = "".join(c if c.isalnum() else "_" for c in representation.lower()).strip("_")
    result.export(RESULTS_DIR / slug, config=config.model_dump(), formats=("csv",))

temporal_scores = pd.concat(temporal_frames, ignore_index=True)
fold_scores = pd.concat(fold_frames, ignore_index=True)
fit_diagnostics = pd.concat(diagnostic_frames, ignore_index=True)

Running Sensors ...


Lightweight assessment failed for Logistic regression: No scalar predictions found for post-hoc assessment.


Running Shared PCA (30) ...


Lightweight assessment failed for Logistic regression: No scalar predictions found for post-hoc assessment.


Running Unaligned PCA (30C) ...


...

Lightweight assessment failed for Logistic regression: No scalar predictions found for post-hoc assessment.


Running Aligned PCA (30C) ...


Lightweight assessment failed for Logistic regression: No scalar predictions found for post-hoc assessment.


Running Aligned PCA (30C, calibration) ...


Lightweight assessment failed for Logistic regression: No scalar predictions found for post-hoc assessment.


### Audit the generated splits

Configuration is not proof. We derive a fold audit from the stored splits and verify that every fold contains exactly one test participant, with no participant appearing in both training and test. Every representation uses the identical outer splitter, so auditing one is enough.

In [34]:
split_rows = results["Sensors"].get_splits()
audit_records = []
for fold in sorted(split_rows["Fold"].unique()):
    fold_rows = split_rows[split_rows["Fold"] == fold]
    train_rows = fold_rows[fold_rows["Set"] == "train"]
    test_rows = fold_rows[fold_rows["Set"] == "test"]
    train_subjects = sorted(train_rows["Group"].astype(str).unique())
    test_subjects = sorted(test_rows["Group"].astype(str).unique())
    overlap = sorted(set(train_subjects) & set(test_subjects))
    audit_records.append({
        "Fold": int(fold),
        "held_out_subject": ", ".join(test_subjects),
        "n_train_subjects": len(train_subjects),
        "n_test_subjects": len(test_subjects),
        "n_train_trials": len(train_rows),
        "n_test_trials": len(test_rows),
        "subject_overlap": ", ".join(overlap),
        "leakage_free": len(overlap) == 0 and len(test_subjects) == 1,
    })

split_audit = pd.DataFrame(audit_records)
if not split_audit["leakage_free"].all():
    raise RuntimeError("The outer-fold audit found participant overlap.")
split_audit

,Fold,held_out_subject,n_train_subjects,n_test_subjects,n_train_trials,n_test_trials,subject_overlap,leakage_free
0,0,001,9,1,405,45,,True
1,1,002,9,1,405,45,,True
2,2,003,9,1,405,45,,True
3,3,004,9,1,405,45,,True
4,4,005,9,1,405,45,,True
5,5,006,9,1,405,45,,True
6,6,007,9,1,405,45,,True
7,7,008,9,1,405,45,,True
8,8,009,9,1,405,45,,True
9,9,010,9,1,405,45,,True


## Step 8. Inspect Time-Resolved Cross-Participant Generalization

The line is mean balanced accuracy across held-out participants; the ribbon is the fold-level standard deviation. The dotted horizontal line marks chance and the vertical line marks movement-cue onset.

Read the *shape* before the maximum. A broad, sustained lift beginning shortly after the cue is the signature of a real motor effect; a single spike surrounded by chance-level noise is what a lucky fold looks like.

In [35]:
temporal_figure = plot_temporal_score_curve(
    temporal_scores[temporal_scores["Representation"].isin(REPRESENTATION_NAMES)],
    metric="balanced_accuracy",
    title=f"{' versus '.join(TARGET_NAMES)}: LOSO decoding",
    colors=REPRESENTATION_COLORS,
)
temporal_figure.add_hline(y=CHANCE_LEVEL, line_dash="dot", line_color="#777777")
temporal_figure.add_vline(x=0, line_color="#999999")
temporal_figure.update_yaxes(title_text="balanced accuracy")
temporal_figure.update_xaxes(title_text="time from movement cue (s)")
temporal_figure.show()

In [36]:
peak_summary = temporal_scores.loc[
    temporal_scores.groupby("Representation")["Mean"].idxmax(),
    ["Representation", "Time", "Mean", "Std"],
].sort_values("Mean", ascending=False)
peak_summary = peak_summary.rename(columns={
    "Time": "peak_time_s",
    "Mean": "peak_balanced_accuracy",
    "Std": "peak_fold_std",
})
peak_summary.round(3)

,Representation,peak_time_s,peak_balanced_accuracy,peak_fold_std
288,Shared PCA (30),0.394,0.729,0.097
95,Sensors,0.394,0.696,0.077
728,Aligned PCA (3C),0.731,0.610,0.072
904,Aligned PCA (30C),0.625,0.574,0.129
510,Unaligned PCA (30C),0.575,0.568,0.095
1134,"Aligned PCA (30C, calibration)",0.856,0.549,0.076


### Gain through time

The curves above are easy to over-read: four of them overlap and the eye is drawn to whichever happens to be on top at the maximum. The plot below re-expresses the same folds as a **paired difference against Sensors** at every latency — the mean and SEM across held-out participants of `(representation − Sensors)` balanced accuracy.

Pairing matters. Participants differ enormously in how decodable they are, and that between-participant variance dominates the ribbons in the previous figure while cancelling exactly in the difference. A curve sitting above zero is a sustained enhancement over raw sensors at that latency; below zero is a cost.

In [37]:
metric_rows = fold_scores[
    (fold_scores["Metric"] == "balanced_accuracy") & fold_scores["Time"].notna()
]
wide_scores = metric_rows.pivot_table(
    index=["Fold", "Time"], columns="Representation", values="Value"
)

gain_frames = []
for representation in OTHER_REPRESENTATIONS:
    gain = (wide_scores[representation] - wide_scores["Sensors"]).rename("Value").reset_index()
    summary = gain.groupby("Time")["Value"].agg(["mean", "std", "count"]).reset_index()
    summary["Model"] = representation
    summary["Metric"] = "balanced_accuracy_gain"
    summary["Mean"] = summary["mean"]
    summary["Std"] = summary["std"].fillna(0.0) / np.sqrt(summary["count"].clip(lower=1))
    gain_frames.append(summary[["Model", "Metric", "Time", "Mean", "Std"]])
gain_through_time = pd.concat(gain_frames, ignore_index=True)

gain_figure = plot_temporal_score_curve(
    gain_through_time,
    metric="balanced_accuracy_gain",
    title="Gain through time: representation minus Sensors (paired by held-out participant)",
    colors=REPRESENTATION_COLORS,
)
gain_figure.add_hline(y=0.0, line_dash="dot", line_color="#777777")
gain_figure.add_vline(x=0, line_color="#999999")
gain_figure.update_yaxes(title_text="balanced accuracy gain over Sensors")
gain_figure.update_xaxes(title_text="time from movement cue (s)")
gain_figure.show()

## Step 9. Examine Held-Out-Participant Variability

A group mean can hide participants with qualitatively different decoding profiles, and with one fold per participant we can simply look at all of them. We retain one temporal curve per LOSO fold, labelled by the held-out participant, and reduce each fold to two predeclared post-cue scalars:

1. **Post-cue mean balanced accuracy:** average accuracy from 0 s to the end of the epoch.
2. **AUC above chance:** the temporal integral of `(balanced accuracy − chance)` over the same interval.

Representation-minus-Sensors differences stay paired within the same held-out participant. These are descriptive consistency diagnostics — the question "is this difference larger than chance?" is deferred to Step 10.

In [38]:
held_out_by_fold = split_audit.set_index("Fold")["held_out_subject"].to_dict()
fold_summary_records = []
for (representation, fold), rows in metric_rows.groupby(["Representation", "Fold"]):
    rows = rows.sort_values("Time")
    active_rows = rows[rows["Time"] >= 0]
    peak_index = rows["Value"].idxmax()
    fold_summary_records.append({
        "Representation": representation,
        "Fold": int(fold),
        "held_out_subject": held_out_by_fold[int(fold)],
        "peak_time_s": float(rows.loc[peak_index, "Time"]),
        "peak_balanced_accuracy": float(rows.loc[peak_index, "Value"]),
        "postcue_mean_balanced_accuracy": float(active_rows["Value"].mean()),
        "postcue_auc_above_chance": float(np.trapezoid(
            active_rows["Value"] - CHANCE_LEVEL, active_rows["Time"]
        )),
    })

fold_summary = pd.DataFrame(fold_summary_records)
representation_summary = (
    fold_summary.groupby("Representation")
    .agg(
        n_folds=("Fold", "nunique"),
        peak_ba_mean=("peak_balanced_accuracy", "mean"),
        peak_ba_std=("peak_balanced_accuracy", "std"),
        postcue_ba_mean=("postcue_mean_balanced_accuracy", "mean"),
        postcue_ba_std=("postcue_mean_balanced_accuracy", "std"),
        postcue_auc_mean=("postcue_auc_above_chance", "mean"),
        postcue_auc_std=("postcue_auc_above_chance", "std"),
    )
    .reset_index()
)
representation_summary.round(3)

,Representation,n_folds,peak_ba_mean,peak_ba_std,postcue_ba_mean,postcue_ba_std,postcue_auc_mean,postcue_auc_std
0,Aligned PCA (30C),10,0.720,0.047,0.504,0.043,0.004,0.043
1,"Aligned PCA (30C, calibration)",10,0.686,0.039,0.498,0.036,-0.002,0.036
2,Aligned PCA (3C),10,0.749,0.090,0.538,0.075,0.038,0.075
3,Sensors,10,0.798,0.068,0.579,0.053,0.080,0.053
4,Shared PCA (30),10,0.820,0.056,0.591,0.054,0.091,0.054
5,Unaligned PCA (30C),10,0.705,0.055,0.491,0.043,-0.009,0.043


In [39]:
paired = fold_summary.pivot(
    index=["Fold", "held_out_subject"],
    columns="Representation",
    values=[
        "peak_balanced_accuracy",
        "postcue_mean_balanced_accuracy",
        "postcue_auc_above_chance",
    ],
)
paired_fold_differences = paired.index.to_frame(index=False)
for metric in (
    "peak_balanced_accuracy",
    "postcue_mean_balanced_accuracy",
    "postcue_auc_above_chance",
):
    for representation in OTHER_REPRESENTATIONS:
        slug = "".join(
            c if c.isalnum() else "_" for c in representation.lower()
        ).strip("_")
        paired_fold_differences[f"{metric}_{slug}_minus_sensors"] = (
            paired[(metric, representation)].to_numpy()
            - paired[(metric, "Sensors")].to_numpy()
        )
paired_fold_differences.round(3)

,Fold,held_out_subject,peak_balanced_accuracy_shared_pca__30_minus_sensors,peak_balanced_accuracy_unaligned_pca__30c_minus_sensors,peak_balanced_accuracy_aligned_pca__3c_minus_sensors,peak_balanced_accuracy_aligned_pca__30c_minus_sensors,postcue_mean_balanced_accuracy_shared_pca__30_minus_sensors,postcue_mean_balanced_accuracy_unaligned_pca__30c_minus_sensors,postcue_mean_balanced_accuracy_aligned_pca__3c_minus_sensors,postcue_mean_balanced_accuracy_aligned_pca__30c_minus_sensors,postcue_auc_above_chance_shared_pca__30_minus_sensors,postcue_auc_above_chance_unaligned_pca__30c_minus_sensors,postcue_auc_above_chance_aligned_pca__3c_minus_sensors,postcue_auc_above_chance_aligned_pca__30c_minus_sensors
0,0,001,-0.002,-0.160,-0.160,-0.158,-0.005,-0.154,-0.127,-0.154,-0.005,-0.155,-0.127,-0.154
1,1,002,0.043,0.000,-0.020,-0.042,0.023,-0.028,-0.074,-0.021,0.023,-0.028,-0.075,-0.020
2,2,003,0.023,-0.222,-0.043,-0.178,0.013,-0.213,0.007,-0.127,0.013,-0.214,0.008,-0.127
3,3,004,0.023,0.002,-0.042,-0.087,0.021,-0.107,-0.045,-0.142,0.021,-0.107,-0.045,-0.143
4,4,005,0.067,-0.111,-0.043,-0.023,0.022,-0.040,-0.010,-0.017,0.022,-0.040,-0.011,-0.018
5,5,006,0.024,-0.088,0.069,-0.042,0.055,-0.029,0.053,0.009,0.056,-0.029,0.053,0.009
6,6,007,0.085,-0.067,0.022,0.024,-0.006,-0.062,-0.024,-0.030,-0.006,-0.062,-0.024,-0.030
7,7,008,0.003,0.003,-0.089,0.001,-0.002,-0.042,-0.117,-0.047,-0.001,-0.042,-0.117,-0.046
8,8,009,-0.022,-0.156,-0.045,-0.111,0.003,-0.096,-0.032,-0.088,0.003,-0.097,-0.031,-0.088
9,9,010,-0.021,-0.122,-0.134,-0.155,-0.006,-0.111,-0.045,-0.139,-0.007,-0.113,-0.047,-0.140


In [40]:
# One heatmap per representation, laid out as a row.
fold_panels = {}
for representation in REPRESENTATION_NAMES:
    rows = metric_rows[metric_rows["Representation"] == representation]
    matrix = rows.pivot(index="Fold", columns="Time", values="Value").sort_index()
    fold_panels[representation] = plot_heatmap(
        matrix.to_numpy(),
        x_labels=matrix.columns.to_numpy(dtype=float),
        y_labels=[f"sub-{held_out_by_fold[int(fold)]}" for fold in matrix.index],
        vmin=0,
        vmax=1,
        title=representation,
        xaxis_title="time from movement cue (s)",
        yaxis_title="held-out participant",
        colorbar_label="BA",
    )

fold_heatmaps = facet_figures(
    fold_panels,
    n_cols=len(REPRESENTATION_NAMES),
    title="Balanced accuracy for every held-out participant",
    row_height=max(420, 32 * len(ANALYZED_SUBJECTS) + 200),
)
fold_heatmaps.show()


In [41]:
# Box + points per representation, one panel per predeclared summary.
distribution_panels = {}
for metric, pretty in (
    ("postcue_mean_balanced_accuracy", "Post-cue mean balanced accuracy"),
    ("postcue_auc_above_chance", "Post-cue AUC above chance"),
):
    distribution_panels[pretty] = plot_distribution_groups(
        groups=[
            fold_summary.loc[
                fold_summary["Representation"] == representation, metric
            ].to_numpy()
            for representation in REPRESENTATION_NAMES
        ],
        labels=REPRESENTATION_NAMES,
        color=[REPRESENTATION_COLORS[name] for name in REPRESENTATION_NAMES],
        show_points=True,
        baseline=0.0 if metric.endswith("auc_above_chance") else CHANCE_LEVEL,
        yaxis_title=pretty,
        title=pretty,
    )

fold_summary_figure = facet_figures(
    distribution_panels,
    n_cols=2,
    title="Per-fold distributions by representation",
    row_height=460,
    shared_yaxes=False,
)
fold_summary_figure.show()


The final view of the same folds puts one dot per held-out participant next to each representation's mean and standard error. This is the plot to consult when a group mean and a paired difference disagree: it shows immediately whether a representation wins on most participants or on one dramatic outlier.

In [42]:
representation_comparison_figure = plot_group_scatter_with_mean(
    [
        fold_summary.loc[
            fold_summary["Representation"] == representation, "peak_balanced_accuracy"
        ].to_numpy()
        for representation in REPRESENTATION_NAMES
    ],
    REPRESENTATION_NAMES,
    point_labels=[
        fold_summary.loc[
            fold_summary["Representation"] == representation, "held_out_subject"
        ].to_numpy()
        for representation in REPRESENTATION_NAMES
    ],
    title="Peak balanced accuracy by representation, one point per held-out participant",
    yaxis_title="peak balanced accuracy",
    baseline=CHANCE_LEVEL,
    baseline_label="chance",
    color=[REPRESENTATION_COLORS[name] for name in REPRESENTATION_NAMES],
    height=520,
)
representation_comparison_figure.show()

## Step 10. Statistical Significance

Everything so far has been descriptive. Now we ask whether the differences survive a null.

The test is a **paired permutation test** on the already-fitted predictions. Within each held-out participant, the predictions from the two representations are randomly swapped — that is the null hypothesis "the representation label is arbitrary" made concrete — and the observed balanced-accuracy difference is compared against the resulting distribution.

Two features make this the right test here:

- **It is paired.** Swapping happens *within* a participant, so the enormous between-participant variance in decodability never enters the null.
- **It is corrected across time.** `temporal_correction='max_stat'` takes the maximum of each permutation's null across all latencies, so a single corrected p-value protects the entire time course rather than one hand-picked peak.

Three comparisons carry the argument:

| Comparison | Question |
|---|---|
| each representation vs **Sensors** | is this representation different from decoding raw channels? |
| Aligned PCA vs **Shared PCA** | does a *participant-specific* basis add anything to a shared one? |
| Aligned PCA vs **Unaligned PCA** | is it the **rotation**, or merely having a per-participant basis? |

The last row is the one that could explain the result away. If unaligned per-participant PCA already performed like the aligned version, the rotation would be decoration.

In [43]:
stats_config = StatisticalAssessmentConfig(
    chance=ChanceAssessmentConfig(
        n_permutations=N_PERMUTATIONS, temporal_correction="max_stat"
    ),
    # The confidence interval's resampling budget; tied to the permutation count
    # so that a reduced run is cheap on both.
    n_bootstraps=N_PERMUTATIONS,
    unit_of_inference="group_mean",
    random_state=SEED,
    n_jobs=N_JOBS,
)
comparison_pairs = [
    *((representation, "Sensors") for representation in OTHER_REPRESENTATIONS),
    (ALIGNED_LARGE, SHARED_PCA),
    (ALIGNED_LARGE, UNALIGNED_PCA),
    (CALIBRATION, SHARED_PCA),
]

significance_frames = []
significance_figures = {}
for representation, baseline in comparison_pairs:
    comparison = run_paired_permutation_assessment(
        results[representation],
        results[baseline],
        "Logistic regression",
        "balanced_accuracy",
        stats_config,
    )
    comparison["Representation"] = representation
    comparison["Baseline"] = baseline
    significance_frames.append(comparison)
    key = f"{representation} vs {baseline}"
    significance_figures[key] = plot_temporal_statistical_assessment(
        comparison,
        title=f"{key}: paired permutation test ({N_PERMUTATIONS} shuffles, max-stat corrected)",
    )

significance_assessment = pd.concat(significance_frames, ignore_index=True)
significance_summary = (
    significance_assessment.loc[
        significance_assessment.groupby(["Representation", "Baseline"])["Observed"].idxmax()
    ]
    .reset_index(drop=True)
    .rename(columns={"Observed": "peak_observed_diff"})
)
significance_summary.round(4)

,Model,Metric,Comparison,peak_observed_diff,InferentialUnit,NEff,NullMethod,NPermutations,P0,PValue,...,NullMedian,NullLower,NullUpper,Significant,Caveat,Time,TrainTime,TestTime,Representation,Baseline
0,Logistic regression,balanced_accuracy,Paired Difference (A-B),0.0934,Group,10,paired_permutation,200,0.0021,0.0498,...,0.0021,-0.0674,0.0852,False,Independence assumed at the 'Group' level.,0.0438,None,None,Aligned PCA (30C),Sensors
1,Logistic regression,balanced_accuracy,Paired Difference (A-B),0.0778,Group,10,paired_permutation,200,0.0018,0.0199,...,0.0018,-0.0687,0.0602,False,Independence assumed at the 'Group' level.,-0.0375,None,None,Aligned PCA (30C),Shared PCA (30)
2,Logistic regression,balanced_accuracy,Paired Difference (A-B),0.1269,Group,10,paired_permutation,200,0.0000,0.0100,...,0.0000,-0.1089,0.1000,True,Independence assumed at the 'Group' level.,-0.0375,None,None,Aligned PCA (30C),Unaligned PCA (30C)
3,Logistic regression,balanced_accuracy,Paired Difference (A-B),0.0911,Group,10,paired_permutation,200,0.0024,0.0498,...,0.0024,-0.0776,0.0781,False,Independence assumed at the 'Group' level.,-0.0312,None,None,"Aligned PCA (30C, calibration)",Shared PCA (30)
4,Logistic regression,balanced_accuracy,Paired Difference (A-B),0.0913,Group,10,paired_permutation,200,0.0021,0.1095,...,0.0021,-0.0781,0.1002,False,Independence assumed at the 'Group' level.,0.0500,None,None,Aligned PCA (3C),Sensors
5,Logistic regression,balanced_accuracy,Paired Difference (A-B),0.0733,Group,10,paired_permutation,200,0.0066,0.0398,...,0.0066,-0.0645,0.0643,False,Independence assumed at the 'Group' level.,-0.0062,None,None,Shared PCA (30),Sensors
6,Logistic regression,balanced_accuracy,Paired Difference (A-B),0.1088,Group,10,paired_permutation,200,-0.0020,0.0199,...,-0.0020,-0.0687,0.0910,False,Independence assumed at the 'Group' level.,-0.0062,None,None,Unaligned PCA (30C),Sensors


In [44]:
for figure in significance_figures.values():
    figure.show()

<div class="alert alert-warning">
<b>⚠️ Read the p-value floor, not the decimal:</b><br>
A permutation p-value cannot go below <code>1 / (n_permutations + 1)</code>. With 200 shuffles that floor is 0.005, and a printed "0.00498" is the floor rather than an estimate of how small the true value is. Report such a result as <b>p &lt; 0.005</b>.
</div>

## Step 11. Bound the Alignment Result

A significant difference is not yet an interpretable one. Three further checks bound what any alignment gain can mean.

### 11a. The within-participant ceiling

LOSO asks a harder question than "can this signal be classified at all" — it demands that the pattern transfer between heads. Refitting the *identical* sliding decoder inside each participant with stratified k-fold cross-validation separates the two failure modes:

- if the within-participant curve is also near chance, the signal is weak;
- if the within-participant curve is high while LOSO is near chance, the signal is strong but **subject-specific**, and the entire cross-participant problem is one of correspondence rather than of signal-to-noise.

That distinction is precisely what alignment is trying to fix, so this curve is the ceiling every representation is working toward.

In [45]:
within_config = sensor_config.model_copy(deep=True)
within_config.cv = CVConfig(
    strategy="stratified", n_splits=WITHIN_SUBJECT_SPLITS, shuffle=True, random_state=SEED
)

within_frames = []
for subject in ANALYZED_SUBJECTS:
    rows = subject_ids == subject
    _, counts = np.unique(y[rows], return_counts=True)
    if len(counts) < 2 or counts.min() < WITHIN_SUBJECT_SPLITS:
        warnings.warn(f"Skipping {subject}: too few trials per class.", stacklevel=2)
        continue
    within_result = Experiment(within_config).run(
        X[rows], y[rows], sample_ids=trial_ids[rows],
        observation_level="epoch", time_axis=times,
    )
    curve = within_result.get_temporal_score_summary()
    curve["subject"] = subject
    within_frames.append(curve)

within_subject_scores = pd.concat(within_frames, ignore_index=True)
within_subject_scores = within_subject_scores[
    within_subject_scores["Metric"] == "balanced_accuracy"
]
within_subject_curve = (
    within_subject_scores.groupby("Time")["Mean"].agg(["mean", "std", "count"]).reset_index()
)
within_subject_curve["Model"] = "Within participant (sensors)"
within_subject_curve["Metric"] = "balanced_accuracy"
within_subject_curve["Mean"] = within_subject_curve["mean"]
within_subject_curve["Std"] = within_subject_curve["std"].fillna(0.0) / np.sqrt(
    within_subject_curve["count"].clip(lower=1)
)
within_subject_curve = within_subject_curve[["Model", "Metric", "Time", "Mean", "Std"]]

upper_bound_figure = plot_temporal_score_curve(
    pd.concat(
        [
            temporal_scores.loc[
                temporal_scores["Representation"] == "Sensors",
                ["Model", "Metric", "Time", "Mean", "Std"],
            ],
            within_subject_curve,
        ],
        ignore_index=True,
    ),
    metric="balanced_accuracy",
    title="Within-participant ceiling versus cross-participant transfer (sensors)",
    colors=REPRESENTATION_COLORS,
)
upper_bound_figure.add_hline(y=CHANCE_LEVEL, line_dash="dot", line_color="#777777")
upper_bound_figure.add_vline(x=0, line_color="#999999")
upper_bound_figure.update_yaxes(title_text="balanced accuracy")
upper_bound_figure.update_xaxes(title_text="time from movement cue (s)")
upper_bound_figure.show()

within_subject_peaks = (
    within_subject_scores.loc[
        within_subject_scores.groupby("subject")["Mean"].idxmax(),
        ["subject", "Time", "Mean"],
    ]
    .rename(columns={"Time": "peak_time_s", "Mean": "peak_balanced_accuracy"})
    .sort_values("subject")
    .reset_index(drop=True)
)
within_subject_peaks.round(3)

,subject,peak_time_s,peak_balanced_accuracy
0,001,0.438,0.850
1,002,0.731,0.835
2,003,0.444,0.895
3,004,0.375,0.825
4,005,0.319,0.840
5,006,0.206,0.850
6,007,0.256,0.745
7,008,0.419,0.830
8,009,0.575,0.915
9,010,0.450,0.710


### 11b. What the rotation actually does, geometrically

The decoder only reports *whether* alignment helped. Here we refit the alignment on every fold and read it out directly, with no classifier involved, using `TemporalProcrustesAlignment` on its own.

For each participant we record how well their grand-mean trajectory matched the training template **before** the rotation and **after** it, on a scale where 1 is a perfect match and 0 is orthogonal. The held-out participant is the row of interest: theirs is the mapping estimated without any labels, which is the one the decoding result depends on.

A large rise means the participant's principal axes pointed somewhere idiosyncratic and the rotation recovered the correspondence. A flat pair means either that the axes already agreed — in which case alignment has nothing to add — or that the grand-mean paths are too dissimilar in shape for any rotation to reconcile.

To keep an $O(\text{participants}^2)$ diagnostic affordable we use up to `GEOMETRY_MAX_TRIALS` trials per participant; the grand-mean path is stable well below the full trial count.

In [46]:
rng = np.random.default_rng(SEED)
keep = []
for subject in ANALYZED_SUBJECTS:
    rows = np.flatnonzero(subject_ids == subject)
    if len(rows) > GEOMETRY_MAX_TRIALS:
        rows = rng.choice(rows, size=GEOMETRY_MAX_TRIALS, replace=False)
    keep.append(rows)
keep = np.sort(np.concatenate(keep))
X_geometry, subjects_geometry = X[keep], subject_ids[keep]

geometry_records = []
for held_out in ANALYZED_SUBJECTS:
    train = subjects_geometry != held_out
    aligner = TemporalProcrustesAlignment(n_components=N_COMPONENTS, random_state=SEED)
    aligner.fit(X_geometry[train], groups=subjects_geometry[train])
    aligner.transform(X_geometry[~train], groups=subjects_geometry[~train])
    for subject, diagnostics in aligner.alignment_diagnostics_.items():
        geometry_records.append({
            "held_out_subject": held_out,
            "subject": subject,
            "role": "held out" if subject == held_out else "training",
            **diagnostics,
        })

alignment_geometry = pd.DataFrame(geometry_records).drop(columns=["seen_in_training"])
held_out_geometry = alignment_geometry[alignment_geometry["role"] == "held out"]

geometry_figure = plot_group_scatter_with_mean(
    [
        held_out_geometry["template_similarity_unrotated"].to_numpy(),
        held_out_geometry["template_similarity_rotated"].to_numpy(),
    ],
    ["before rotation", "after rotation"],
    point_labels=[
        held_out_geometry["subject"].to_numpy(),
        held_out_geometry["subject"].to_numpy(),
    ],
    title="Shape agreement between each held-out participant's mean path and the training template",
    yaxis_title="normalized similarity to template",
    baseline=0.0,
    color=[REPRESENTATION_COLORS[UNALIGNED_PCA], REPRESENTATION_COLORS[ALIGNED_LARGE]],
    height=460,
)
geometry_figure.show()

held_out_geometry.round(3)

,held_out_subject,subject,role,template_similarity_unrotated,template_similarity_rotated,similarity_gain,procrustes_disparity,rotation_angle_deg
9,001,001,held out,-0.162,0.839,1.001,0.161,90.968
19,002,002,held out,0.299,0.677,0.378,0.323,91.376
29,003,003,held out,-0.521,0.793,1.314,0.207,92.250
39,004,004,held out,-0.241,0.867,1.108,0.133,93.809
49,005,005,held out,0.092,0.603,0.511,0.397,87.328
59,006,006,held out,0.292,0.734,0.442,0.266,90.345
69,007,007,held out,0.355,0.673,0.318,0.327,89.451
79,008,008,held out,0.564,0.793,0.229,0.207,86.468
89,009,009,held out,-0.099,0.652,0.752,0.348,93.241
99,010,010,held out,0.275,0.821,0.546,0.179,88.871


### 11c. Is the gain just transduction?

Transductive alignment estimates the held-out participant's PCA and rotation from *all* of their unlabeled trials — including the very trials it is then scored on. No labels leak, but data does, and a sceptical reader is entitled to ask whether that is doing the work.

The calibration variant removes the concern by cross-fitting. Each half of the held-out participant's trials is mapped using the PCA and rotation estimated from the **other** half, so no trial ever informs the mapping applied to it, while every trial is still scored.

- If the transductive and calibration curves agree, the gain is genuine unsupervised domain adaptation from a calibration batch.
- If the calibration curve collapses back onto Shared PCA, the apparent gain was the transduction.

In [47]:
calibration_figure = plot_temporal_score_curve(
    temporal_scores[
        temporal_scores["Representation"].isin([SHARED_PCA, ALIGNED_LARGE, CALIBRATION])
    ],
    metric="balanced_accuracy",
    title="Transductive alignment versus calibration-half alignment",
    colors=REPRESENTATION_COLORS,
)
calibration_figure.add_hline(y=CHANCE_LEVEL, line_dash="dot", line_color="#777777")
calibration_figure.add_vline(x=0, line_color="#999999")
calibration_figure.update_yaxes(title_text="balanced accuracy")
calibration_figure.update_xaxes(title_text="time from movement cue (s)")
calibration_figure.show()

peak_summary[peak_summary["Representation"].isin(
    [SHARED_PCA, ALIGNED_LARGE, CALIBRATION]
)].round(3)

,Representation,peak_time_s,peak_balanced_accuracy,peak_fold_std
288,Shared PCA (30),0.394,0.729,0.097
904,Aligned PCA (30C),0.625,0.574,0.129
1134,"Aligned PCA (30C, calibration)",0.856,0.549,0.076


## Step 12. Compare Contrasts Side by Side

The 12 steps above answer one question: **left versus right hand execution**. Several of the most useful claims, though, are *comparisons between* contrasts — whether imagery behaves like execution, whether imagination and execution are separable at all, and how a four-class target compares to the binaries. Those need all four panels in one view.

Only the **temporal decoding** is repeated for the other contrasts. The three validations in Step 11 bound the *alignment* claim rather than the comparison between contrasts, so they stay with the primary contrast. Even so this re-runs five representations three more times — set `EEG_CONTRAST_COMPARISON=0` to skip it during a quick local pass.


In [48]:
COMPARISON_ACTIVE_WINDOW = (0.0, 1.0)  # post-cue interval, as in Step 9


def decode_contrast(conditions):
    """Decode one contrast across the five representations.

    Returns the fold-averaged time courses and the fold-level scores. Both are
    needed: the curves give the reported summaries, the folds give the spread
    across held-out participants that the figures draw.
    """
    container = load_eegbci_container(
        BIDS_ROOT,
        subjects=SUBJECTS,
        runs=tuple(range(3, 15)),
        conditions=conditions,
        tmin=ANALYSIS_WINDOW[0],
        tmax=ANALYSIS_WINDOW[1],
        baseline=(-0.2, 0.0),
    )
    features = np.asarray(container.X, dtype=np.float32)
    time_axis = np.asarray(container.coords["time"], dtype=float)
    mapping = {value: index for index, value in enumerate(conditions)}
    target = np.array([mapping[value] for value in np.asarray(container.y, dtype=int)])
    groups = np.asarray(container.coords["subject"]).astype(str)
    ids = np.asarray(container.ids).astype(str)
    del container

    temporal_frames, fold_frames = [], []
    for representation, config in experiments.items():
        result = Experiment(config).run(
            features,
            target,
            groups=groups,
            sample_ids=ids,
            observation_level="epoch",
            inferential_unit="subject",
            time_axis=time_axis,
        )
        curve = result.get_temporal_score_summary()
        curve["Representation"] = representation
        curve["Model"] = representation
        temporal_frames.append(curve)

        folds = result.get_detailed_scores()
        folds["Representation"] = representation
        fold_frames.append(folds)
    return (
        pd.concat(temporal_frames, ignore_index=True),
        pd.concat(fold_frames, ignore_index=True),
    )


def contrast_fold_metrics(curves, folds, chance):
    """Per-fold values behind each reported summary, for one contrast.

    The peak is read off the **fold-averaged** curve and every fold is then
    sampled at that one latency, so the points average exactly to the reported
    number. Taking each fold's own maximum instead would select a different
    latency per fold and inflate the mean — the maximum of noisy estimates is
    not an estimate of the maximum.

    The two window summaries are averages over a fixed interval, so averaging
    over folds first or over time first gives the same value; the fold level is
    kept only for the spread.
    """
    rows = folds[(folds["Metric"] == "balanced_accuracy") & folds["Time"].notna()]
    records = []
    for representation, group in rows.groupby("Representation"):
        curve = curves[curves["Representation"] == representation].sort_values("Time")
        peak_time = float(curve.loc[curve["Mean"].idxmax(), "Time"])
        for fold, fold_rows in group.groupby("Fold"):
            fold_rows = fold_rows.sort_values("Time")
            at_peak = fold_rows.loc[
                (fold_rows["Time"] - peak_time).abs().idxmin(), "Value"
            ]
            active = fold_rows[
                (fold_rows["Time"] >= COMPARISON_ACTIVE_WINDOW[0])
                & (fold_rows["Time"] <= COMPARISON_ACTIVE_WINDOW[1])
            ]
            records.append(
                {
                    "Representation": representation,
                    "Fold": int(fold),
                    "peak_time_s": peak_time,
                    "peak_balanced_accuracy": float(at_peak),
                    "postcue_mean_balanced_accuracy": float(active["Value"].mean()),
                    "postcue_auc_above_chance": float(
                        np.trapezoid(active["Value"] - chance, active["Time"])
                    ),
                }
            )
    return pd.DataFrame(records)


COMPARISON_METRICS = {
    "peak_balanced_accuracy": "Peak balanced accuracy",
    "postcue_mean_balanced_accuracy": "Post-cue mean balanced accuracy",
    "postcue_auc_above_chance": "Post-cue AUC above chance",
}

contrast_panels = {}
contrast_folds = {}
contrast_scalars = []

if RUN_CONTRAST_COMPARISON:
    for conditions in COMPARISON_CONTRASTS:
        label = contrast_label(conditions, LABEL_NAMES, short=True)
        chance = 1.0 / len(conditions)
        if tuple(conditions) == tuple(CONDITIONS):
            curves, folds = temporal_scores, fold_scores  # already decoded above
        else:
            print(f"Decoding {label} ...")
            curves, folds = decode_contrast(tuple(conditions))
        curves = curves[curves["Representation"].isin(REPRESENTATION_NAMES)]
        folds = folds[folds["Representation"].isin(REPRESENTATION_NAMES)]

        panel = plot_temporal_score_curve(
            curves,
            metric="balanced_accuracy",
            title=label,
            colors=REPRESENTATION_COLORS,
        )
        panel.add_hline(y=chance, line_dash="dot", line_color="#777777")
        panel.add_vline(x=0, line_color="#999999")
        panel.update_yaxes(title_text="balanced accuracy")
        panel.update_xaxes(title_text="time from movement cue (s)")
        contrast_panels[label] = panel

        per_fold = contrast_fold_metrics(curves, folds, chance)
        per_fold.insert(0, "Contrast", label)
        contrast_folds[label] = per_fold

        summary = per_fold.groupby("Representation", as_index=False)[
            ["peak_time_s", *COMPARISON_METRICS]
        ].mean()
        summary.insert(0, "Contrast", label)
        contrast_scalars.append(summary)

    # Chance differs between the binary and multiclass panels, so the y-axes
    # are left independent rather than shared.
    contrast_comparison_figure = facet_figures(
        contrast_panels,
        n_cols=2,
        title="LOSO decoding across contrasts",
        shared_yaxes=False,
    )
    contrast_comparison_figure.show()

    contrast_summary = pd.concat(contrast_scalars, ignore_index=True)
    contrast_fold_table = pd.concat(contrast_folds.values(), ignore_index=True)
    contrast_summary.to_csv(OUTPUT / "contrast_summary.csv", index=False)
    contrast_fold_table.to_csv(OUTPUT / "contrast_fold_metrics.csv", index=False)

    # One figure per scalar, in the same style as Step 9: a point per held-out
    # participant plus the mean and its SEM. Balanced accuracy and AUC keep
    # separate figures because they do not share a scale.
    contrast_metric_figures = {}
    for column, pretty in COMPARISON_METRICS.items():
        panels = {}
        for label, per_fold in contrast_folds.items():
            chance = 1.0 / len(
                next(c for c in COMPARISON_CONTRASTS
                     if contrast_label(c, LABEL_NAMES, short=True) == label)
            )
            panels[label] = plot_group_scatter_with_mean(
                [
                    per_fold.loc[
                        per_fold["Representation"] == representation, column
                    ].to_numpy()
                    for representation in REPRESENTATION_NAMES
                ],
                REPRESENTATION_NAMES,
                point_labels=[
                    per_fold.loc[
                        per_fold["Representation"] == representation, "Fold"
                    ].to_numpy()
                    for representation in REPRESENTATION_NAMES
                ],
                yaxis_title=pretty,
                baseline=0.0 if column.endswith("auc_above_chance") else chance,
                color=[REPRESENTATION_COLORS[name] for name in REPRESENTATION_NAMES],
            )
        figure = facet_figures(
            panels, n_cols=2, title=f"{pretty} across contrasts", shared_yaxes=False
        )
        figure.show()
        contrast_metric_figures[column] = figure

    for figure_name, figure in [
        ("contrast_comparison", contrast_comparison_figure),
        *((f"contrast_{column}", fig) for column, fig in contrast_metric_figures.items()),
    ]:
        figure.write_html(FIGURES_DIR / f"{figure_name}.html", include_plotlyjs="cdn")

    for column, pretty in COMPARISON_METRICS.items():
        print(f"\n{pretty}")
        print(
            contrast_summary.pivot(
                index="Contrast", columns="Representation", values=column
            )
            .round(3)
            .to_string()
        )
    display(contrast_summary.round(3))
else:
    print("Contrast comparison skipped (set EEG_CONTRAST_COMPARISON=1 to enable).")


Decoding Left Hand (Imag) vs Right Hand (Imag) ...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Reading 0 ... 1999


Peak balanced accuracy
Representation                         Aligned PCA (30C)  Aligned PCA (3C)  Sensors  Shared PCA (30)  Unaligned PCA (30C)
Contrast                                                                                                                 
4-class                                            0.289             0.309    0.372            0.384                0.292
Hands (Exec) vs Hands (Imag)                       0.562             0.596    0.598            0.579                0.581
Left Hand (Exec) vs Right Hand (Exec)              0.574             0.610    0.696            0.729                0.568
Left Hand (Imag) vs Right Hand (Imag)              0.584             0.601    0.659            0.673                0.605

Post-cue mean balanced accuracy
Representation                         Aligned PCA (30C)  Aligned PCA (3C)  Sensors  Shared PCA (30)  Unaligned PCA (30C)
Contrast                                                                                 

,Contrast,Representation,peak_time_s,peak_balanced_accuracy,postcue_mean_balanced_accuracy,postcue_auc_above_chance
0,Left Hand (Exec) vs Right Hand (Exec),Aligned PCA (30C),0.625,0.574,0.504,0.004
1,Left Hand (Exec) vs Right Hand (Exec),Aligned PCA (3C),0.731,0.610,0.538,0.038
2,Left Hand (Exec) vs Right Hand (Exec),Sensors,0.394,0.696,0.579,0.080
3,Left Hand (Exec) vs Right Hand (Exec),Shared PCA (30),0.394,0.729,0.591,0.091
4,Left Hand (Exec) vs Right Hand (Exec),Unaligned PCA (30C),0.575,0.568,0.491,-0.009
5,Left Hand (Imag) vs Right Hand (Imag),Aligned PCA (30C),0.181,0.584,0.499,-0.001
6,Left Hand (Imag) vs Right Hand (Imag),Aligned PCA (3C),0.525,0.601,0.524,0.025
7,Left Hand (Imag) vs Right Hand (Imag),Sensors,0.644,0.659,0.579,0.080
8,Left Hand (Imag) vs Right Hand (Imag),Shared PCA (30),0.612,0.673,0.596,0.096
9,Left Hand (Imag) vs Right Hand (Imag),Unaligned PCA (30C),0.250,0.605,0.519,0.019


## Conclusions & Interpretation Checklist

Before making a decoding claim from this analysis, verify the whole chain of evidence:

- **Participant separation:** every outer fold must have zero train/test participant overlap (Step 7's audit raises if not).
- **Fold-local preprocessing:** scaling, PCA and alignment are all fitted after the outer split — nothing that touched the held-out participant's data was fitted before it was set aside.
- **Inferential unit:** folds represent held-out participants, not independent time points or trials, and the permutation test swaps within participants for the same reason.
- **Temporal multiplicity:** a descriptive peak is not a corrected significance test; use the max-stat-corrected result from Step 10 and report the p-value floor honestly.
- **The rotation control:** any claim that alignment helps must be stated against **Unaligned PCA**, not only against Sensors. Without that comparison, "a per-participant basis helps" and "the rotation helps" are indistinguishable.
- **Calibration scope:** aligned PCA consumes unlabeled held-out-participant trials and is transductive; the Step 11c variant is the version that can be described as leakage-free.
- **The ceiling:** read every LOSO number against the within-participant curve. A representation that closes part of that gap is doing something a better classifier could not.

<div class="alert alert-success">
<b>🎯 Main takeaway:</b><br>
Compare representations through <i>paired</i> held-out-participant behaviour and sustained temporal performance, tested against a null and bounded by a control. A single group-level peak establishes nothing on its own.
</div>

## Running the Same Analysis Headlessly

The companion script repeats the same computations, exports the complete `ExperimentResult` objects, and invokes the same report renderer:

```bash
python scripts/analysis_eegbci_decoding.py
```

Useful variations:

```bash
# quick validation run
python scripts/analysis_eegbci_decoding.py --subjects 1 2 3 --n-permutations 20 --n-jobs 1

# a different contrast: left vs right, imagined rather than executed
python scripts/analysis_eegbci_decoding.py --conditions 5 6

# four-class decoding
python scripts/analysis_eegbci_decoding.py --conditions 3 4 5 6

# all four contrasts as one sweep: each lands in its own subdirectory and every
# report figure gains one panel per contrast
python scripts/analysis_eegbci_decoding.py --contrasts 3-4 5-6 7-9 3-4-5-6

# combine contrasts that were run as separate cluster jobs, without decoding
python scripts/analysis_eegbci_decoding.py --report-only
```

Condition ids are 3 = left-hand execution, 4 = right-hand execution, 5 = left-hand imagination, 6 = right-hand imagination, 7 = both-hands execution, 8 = both-feet execution, 9 = both-hands imagination, 10 = both-feet imagination. Two ids give binary decoding; more give multiclass, and chance moves accordingly. Add `--prepare` only when the script should prepare the requested BIDS data first.